In [1]:
import os
import glob
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt

!pip install biopython
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from Bio.PDB.SASA import ShrakeRupley
from scipy.spatial import Delaunay


In [3]:
# Define the project directory
# If you moved the notebook, this falls back to the notebook's current working directory.

structures_dir = Path("/Users/devonfinlay/Downloads/CBB_class_assignment_2")
if not structures_dir.exists():
    structures_dir = Path.cwd()

data_dir = structures_dir / "duplicate_mutant_structures"
assert data_dir.exists(), f"Could not find data directory: {data_dir}"

print(f"Using structures directory: {structures_dir}")


Using structures directory: C:\Users\afcor\LocalDocs\Yale Canvas\CBB_class_assignment_2\CBB_class_assignment_2


# The below code calculates the rSASA between any two decoys. Meant to be used between ligandless and ligand bound!

## Calculate the $\Delta$rSASA for all ligands proposed for the system

In [5]:
parser = PDBParser(QUIET=True)

In [29]:
# Calculate Delta rSASA for all mutants in each system
# rSASA here is computed with Biopython Shrake-Rupley and normalized by residue-specific max ASA values.

three_to_one = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLU": "E", "GLN": "Q", "GLY": "G",
    "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P", "SER": "S",
    "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "ASX": "B", "GLX": "Z", "SEC": "U", "PYL": "O"
}


max_asa = {
    "A": 121.0, "R": 265.0, "N": 187.0, "D": 187.0, "C": 148.0,
    "Q": 214.0, "E": 214.0, "G": 97.0, "H": 216.0, "I": 195.0,
    "L": 191.0, "K": 230.0, "M": 203.0, "F": 228.0, "P": 154.0,
    "S": 143.0, "T": 163.0, "W": 264.0, "Y": 255.0, "V": 165.0,
    "B": 187.0, "Z": 214.0, "X": 200.0
}

def compute_rsasa_from_pdb(pdb_path):
    structure = parser.get_structure("p", pdb_path)
    model = next(structure.get_models())

    sr = ShrakeRupley()
    sr.compute(model, level="R")

    rows = []
    for chain in model:
        for residue in chain:
            if not is_aa(residue, standard=False):
                continue
            if "CA" not in residue:
                continue
            res3 = residue.get_resname().strip()
            res1 = three_to_one.get(res3, "X")
            max_area = max_asa.get(res1, 200.0)
            sasa = float(getattr(residue, "sasa", np.nan))
            rsasa = sasa / max_area if np.isfinite(sasa) and max_area > 0 else np.nan

            _, resseq, icode = residue.id
            rows.append({
                "chain": chain.id,
                "resseq": int(resseq),
                "icode": (icode or "").strip(),
                "resname1": res1,
                "rSASA": rsasa,
            })

    return pd.DataFrame(rows)


In [131]:
#### CALCULATE THE DELTA RSASA HERE ####

def calculate_del_rSASA(default_ligand_model, modified_ligand_model):
    rSASA_values_o = []
    pull_table = compute_rsasa_from_pdb(os.path.join(test_dir, default_ligand_model))['rSASA']
    for j in range(0,len(pull_table)):
        rSASA_values_o.append(pull_table[j])
    rSASA_values_o = np.asarray(rSASA_values_o)
    
    rSASA_values_m = []
    pull_table = compute_rsasa_from_pdb(os.path.join(test_dir, modified_ligand_model))['rSASA']
    for j in range(0,len(pull_table)):
        rSASA_values_m.append(pull_table[j])
    rSASA_values_m = np.asarray(rSASA_values_m)
    
    return (rSASA_values_m - rSASA_values_o[0:len(rSASA_values_m)])


In [91]:
test_dir = structures_dir / "Final_Project_Tests"
print(test_dir)
test_file = os.path.join(test_dir,'2AZ5.pdb')

C:\Users\afcor\LocalDocs\Yale Canvas\CBB_class_assignment_2\CBB_class_assignment_2\Final_Project_Tests


In [93]:
test_df = compute_rsasa_from_pdb(test_file)

In [135]:
del_rSASA = calculate_del_rSASA('2AZ5.pdb','2AZ5_ligandless.pdb')
print(del_rSASA.sum())

3.8878932913753057


## This code is rump code, made to be used as stated with PyMol in the .md file

In [149]:

list_obj_test = [[0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]
len(list_obj_test)

35

## Debugging Code, For Checking Variables As Needed

In [125]:
test_df.loc[45:50]

,chain,resseq,icode,resname1,rSASA
45,A,60,,S,0.000000
46,A,61,,Q,0.055075
47,A,62,,V,0.000000
48,A,63,,L,0.177035
49,A,64,,F,0.009399
50,A,65,,K,0.348217


In [229]:
compute_rsasa_from_pdb(system_data[0]['duplicate_files'][0])['rSASA'][0]

0.45227510238407226